# CO<sub>2</sub> Emissions Regression — Portfolio Project

**Author:** Tim  
**Dataset:** Our World in Data – CO<sub>2</sub> and Greenhouse Gas Emissions  
**Objective:** Predict national CO<sub>2</sub> emissions from energy, economic, and demographic features using multiple regression models.

---

## 1. Introduction

### Problem Statement
CO<sub>2</sub> emissions are the primary driver of anthropogenic climate change. Accurately predicting a country's annual emissions from economic and energy indicators helps policymakers design evidence-based climate targets.

### Goal
Build and compare five regression models to predict a country's annual CO<sub>2</sub> emissions (million tonnes).

**Features used:**
- **Energy:** primary energy consumption, coal, oil, gas, cement, flaring emissions
- **Economic:** GDP
- **Demographic:** population
- **Temporal:** year

**Modeling approach:**
1. Linear Regression (baseline)
2. Ridge Regression (L2 regularization)
3. Lasso Regression (L1 — implicit feature selection)
4. Random Forest (bagging ensemble)
5. Gradient Boosting (boosting ensemble)

> **Target variable:** `log(co2)` — the heavily right-skewed target is log-transformed to stabilise variance and improve linear model performance.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print("Libraries loaded.")

Libraries loaded.


## Loading the CO<sub>2</sub> global Dataset from github Repository

### Dataset Overview
50,411 rows | 79 columns - tracking CO2 and greenhouse gas emissions by country and year.

In [4]:
URL = 'https://raw.githubusercontent.com/owid/co2-data/refs/heads/master/owid-co2-data.csv'
data = pd.read_csv(URL)
print(f"Shape: {data.shape[0]:,} rows x {data.shape[1]} columns")
data.head()

Shape: 50,411 rows x 79 columns


,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,1753,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,1754,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(data.info())
data.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50411 entries, 0 to 50410
Data columns (total 79 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   country                                    50411 non-null  object 
 1   year                                       50411 non-null  int64  
 2   iso_code                                   42480 non-null  object 
 3   population                                 41167 non-null  float64
 4   gdp                                        15251 non-null  float64
 5   cement_co2                                 29173 non-null  float64
 6   cement_co2_per_capita                      25648 non-null  float64
 7   co2                                        29384 non-null  float64
 8   co2_growth_abs                             27216 non-null  float64
 9   co2_growth_prct                            26239 non-null  float64
 10  co2_including_luc     

,count,mean,std,min,25%,50%,75%,max
year,50411.0,1.920349e+03,6.585912e+01,1.750000e+03,1.875000e+03,1.925000e+03,1.975000e+03,2.024000e+03
population,41167.0,6.017453e+07,3.308433e+08,2.150000e+02,3.272140e+05,2.291594e+06,9.986553e+06,8.161973e+09
gdp,15251.0,3.300495e+11,3.086383e+12,4.998000e+07,7.874038e+09,2.743861e+10,1.212627e+11,1.301126e+14
cement_co2,29173.0,7.890109e+00,6.298817e+01,0.000000e+00,0.000000e+00,0.000000e+00,5.240000e-01,1.666885e+03
cement_co2_per_capita,25648.0,6.001252e-02,1.235623e-01,0.000000e+00,0.000000e+00,1.000000e-03,7.625000e-02,2.484000e+00
...,...,...,...,...,...,...,...,...
temperature_change_from_n2o,38280.0,5.090648e-04,3.047886e-03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,8.500000e-02
total_ghg,38150.0,4.907996e+02,2.414077e+03,-1.972500e+01,1.502000e+00,1.460550e+01,7.650850e+01,5.443340e+04
total_ghg_excluding_lucf,37813.0,3.105215e+02,1.812364e+03,0.000000e+00,2.210000e-01,2.222000e+00,2.786300e+01,4.371478e+04
trade_co2,4712.0,-6.986781e+00,2.590182e+02,-2.177807e+03,-2.262250e+00,1.641000e+00,1.142550e+01,1.768846e+03


In [ ]:
print(f"Year range        : {data['year'].min()} - {data['year'].max()}")
print(f"Countries/regions : {data['country'].nunique()}")
print(f"\nTop 15 columns by % missing:")
missing_pct = data.isnull().mean().mul(100).sort_values(ascending=False)
print(missing_pct[missing_pct > 0].head(15).map('{:.1f}%'.format))

## 2. Exploratory Data Analysis

In [ ]:
eda_df = data[data['iso_code'].notna() & (data['year'] >= 1990) & (data['co2'] > 0)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(eda_df['co2'], bins=80, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('CO2 Emissions — Raw Distribution', fontsize=13)
axes[0].set_xlabel('CO2 (million tonnes)')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log(eda_df['co2']), bins=60, color='coral', edgecolor='white', alpha=0.85)
axes[1].set_title('CO2 Emissions — Log Distribution', fontsize=13)
axes[1].set_xlabel('log(CO2)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Target Variable Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Raw skewness : {eda_df['co2'].skew():.2f}")
print(f"Log skewness : {np.log(eda_df['co2']).skew():.2f}")

In [ ]:
latest_year = data[data['iso_code'].notna() & data['co2'].notna()]['year'].max()
top10 = (data[(data['year'] == latest_year) & data['iso_code'].notna() & data['co2'].notna()]
         .nlargest(10, 'co2')[['country', 'co2']])

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10['country'][::-1].values, top10['co2'][::-1].values,
               color='steelblue', edgecolor='white')
ax.bar_label(bars, labels=[f'{v:,.0f}' for v in top10['co2'][::-1].values], padding=5)
ax.set_title(f'Top 10 CO2 Emitters ({latest_year})', fontsize=14, fontweight='bold')
ax.set_xlabel('CO2 Emissions (million tonnes)')
plt.tight_layout()
plt.show()

In [ ]:
latest_year = data[data['iso_code'].notna() & data['co2'].notna()]['year'].max()
top5_countries = (data[(data['year'] == latest_year) & data['iso_code'].notna() & data['co2'].notna()]
                  .nlargest(5, 'co2')['country'].tolist())

trend_df = data[data['country'].isin(top5_countries) & (data['year'] >= 1990)]

fig, ax = plt.subplots(figsize=(12, 6))
for country in top5_countries:
    subset = trend_df[trend_df['country'] == country]
    ax.plot(subset['year'], subset['co2'], marker='o', markersize=2, linewidth=2, label=country)

ax.set_title('CO2 Emissions Trends (1990–Present) — Top 5 Emitters', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('CO2 Emissions (million tonnes)')
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ['co2', 'primary_energy_consumption', 'coal_co2', 'oil_co2',
             'gas_co2', 'cement_co2', 'population', 'gdp', 'flaring_co2']

corr_data = (data[data['iso_code'].notna() & (data['year'] >= 1990)][corr_cols]
             .dropna())

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_data.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

### Filtering Strategy
| Decision | Rationale |
|----------|-----------|
| `iso_code` not null | Keeps country-level rows; excludes regional aggregates (e.g., "World", "Asia") |
| `year >= 1990` | Data coverage is substantially better post-1990 |
| `co2 > 0` | Required for log-transform; avoids undefined `log(0)` |

### Missing Feature Values
Filled with **0** — a missing emission source (e.g., `flaring_co2 = NaN`) genuinely means zero reported activity.

### Target Transformation
`log(co2)` corrects the heavy right-skew and stabilises variance. Metrics are reported on the original scale via `exp`.

### Train / Test Split
80% training — 20% test, `random_state=42`. `StandardScaler` is fit **only on the training set** to prevent data leakage.

In [ ]:
FEATURES = [
    'population', 'gdp', 'primary_energy_consumption',
    'coal_co2', 'oil_co2', 'gas_co2', 'cement_co2',
    'flaring_co2', 'other_industry_co2', 'year'
]
TARGET = 'co2'

df = (data[
    data['iso_code'].notna() &
    (data['year'] >= 1990) &
    (data[TARGET] > 0)
][FEATURES + [TARGET]].copy())

df[FEATURES] = df[FEATURES].fillna(0)
df['log_co2'] = np.log(df[TARGET])
df.dropna(subset=['log_co2'], inplace=True)

print(f"Clean dataset : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nTarget statistics (log scale):")
print(df['log_co2'].describe().round(3))

In [ ]:
X = df[FEATURES]
y = df['log_co2']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training samples : {X_train.shape[0]:,}")
print(f"Test samples     : {X_test.shape[0]:,}")
print(f"Features         : {X_train.shape[1]}")

## 4. Model Training & Evaluation

Predictions are made on the **log scale** then inverse-transformed via `exp` before computing RMSE and MAE on the original scale.

| Metric | Description |
|--------|-------------|
| **R²** | Proportion of variance explained (higher = better, max 1.0) |
| **RMSE** | Root Mean Squared Error in original scale (million tonnes) |
| **MAE** | Mean Absolute Error in original scale (million tonnes) |

In [ ]:
results = {}

def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred_log = model.predict(X_te)

    y_pred = np.exp(y_pred_log)
    y_true = np.exp(y_te)

    r2   = r2_score(y_te, y_pred_log)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)

    results[name] = {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'model': model}
    print(f"{name:<28}  R2={r2:.4f}  RMSE={rmse:>12,.1f}  MAE={mae:>12,.1f}")
    return model

In [ ]:
lr = evaluate(
    'Linear Regression',
    LinearRegression(),
    X_train_sc, X_test_sc, y_train, y_test
)

In [ ]:
ridge = evaluate(
    'Ridge (a=1.0)',
    Ridge(alpha=1.0),
    X_train_sc, X_test_sc, y_train, y_test
)

In [ ]:
lasso = evaluate(
    'Lasso (a=0.01)',
    Lasso(alpha=0.01, max_iter=5000),
    X_train_sc, X_test_sc, y_train, y_test
)

print("\nLasso non-zero coefficients:")
for feat, coef in zip(FEATURES, lasso.coef_):
    if coef != 0:
        print(f"  {feat:<30} {coef:+.4f}")

In [ ]:
rf = evaluate(
    'Random Forest',
    RandomForestRegressor(n_estimators=200, max_depth=15,
                          min_samples_leaf=2, random_state=42, n_jobs=-1),
    X_train, X_test, y_train, y_test
)

In [ ]:
gb = evaluate(
    'Gradient Boosting',
    GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                              max_depth=5, subsample=0.8, random_state=42),
    X_train, X_test, y_train, y_test
)

In [ ]:
comparison = (
    pd.DataFrame(
        {k: {m: v for m, v in vals.items() if m != 'model'} for k, vals in results.items()}
    ).T.sort_values('R2', ascending=False)
)

print("=== Model Performance Summary ===")
print(comparison.to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63', '#9C27B0'][:len(comparison)]
metrics = ['R2', 'RMSE', 'MAE']

for ax, metric in zip(axes, metrics):
    vals = comparison[metric]
    ax.bar(range(len(vals)), vals, color=palette, edgecolor='white', width=0.65)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(vals.index, rotation=30, ha='right', fontsize=9)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    if metric == 'R2':
        ax.set_ylim(0, 1)

plt.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
tree_models = {k: v for k, v in results.items() if hasattr(v['model'], 'feature_importances_')}
best_name = max(tree_models, key=lambda k: tree_models[k]['R2'])

importances = pd.Series(
    results[best_name]['model'].feature_importances_, index=FEATURES
).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
bar_colors = ['#FF5252' if i == importances.idxmax() else 'steelblue'
              for i in importances.index]
bars = ax.barh(importances.index, importances.values, color=bar_colors, edgecolor='white')
ax.bar_label(bars, labels=[f'{v:.3f}' for v in importances.values], padding=3, fontsize=9)
ax.set_title(f'Feature Importances — {best_name}', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
y_pred_log = results[best_name]['model'].predict(X_test)
residuals = y_test.values - y_pred_log

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred_log, residuals, alpha=0.35, color='steelblue', s=12)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title(f'Residuals vs Fitted — {best_name}', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Fitted values (log scale)')
axes[0].set_ylabel('Residuals')

axes[1].hist(residuals, bins=50, color='coral', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Residual Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()
print(f"Mean residual : {residuals.mean():.4f}  (ideally ~0)")
print(f"Std  residual : {residuals.std():.4f}")

## 5. Key Takeaways

### Model Performance
- **Gradient Boosting** and **Random Forest** substantially outperform linear models, capturing non-linear interactions between energy sources, GDP, and population.
- Log-transforming the target significantly improved linear model fits — raw CO₂ is heavily right-skewed (skewness > 20).
- **Lasso** zeroed out low-signal features, confirming that a small set of energy-source columns drives most predictive power.

### Feature Insights
- **Primary energy consumption** and major fossil-fuel sources (coal, oil, gas) dominate feature importance — consistent with domain knowledge that combustion is the primary CO₂ driver.
- **GDP** and **population** add meaningful cross-country signal.

### Potential Improvements
| Area | Suggestion |
|------|-----------|
| Hyperparameter tuning | `GridSearchCV` / `Optuna` for RF & GB |
| Feature engineering | Lagged features, rolling averages, GDP per capita |
| Advanced models | XGBoost, LightGBM, CatBoost |
| Temporal validation | Time-based train/test split to respect ordering |
| Interpretability | SHAP values for per-prediction explanations |

---
*Data source: [Our World in Data – CO₂ and Greenhouse Gas Emissions](https://github.com/owid/co2-data)*